# Data Analysis Pipeline : Clustering

**AIMS**  
1.Understanding the Disease
What is HGSOC, and why does it spread in the peritoneal cavity?
Why is peritoneal carcinomatosis hard to treat?
What is the tumor microenvironment (TME)?
What is platinum resistance?

2.Understanding the Data
What do the data types provide?
Why is spatial context important?
Key features from each dataset?

3. Characterizing the TME
How to identify immune cells?
How to define spatial regions?
Indicators of immune states?
How to detect spatial heterogeneity?

4. Linking to Disease Mechanisms
Patterns explaining resistance?
Tumor–immune interactions?
Patient comparison (hypothetical):

5. Results & Interpretation
Key spatial patterns?
Findings linked to aggressiveness/resistance?
Main biological insight?
Therapeutic implications?

# 0 : Data Loading

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import scimap as sm

ModuleNotFoundError: No module named 'scimap'

In [ ]:
# Load TSV with gene names as index
df = pd.read_csv("wait for outpuf file from space ranger")
adata = sc.AnnData(df.T)

# 1 : Statistics Part

# 2 : Visualization Part

## A) Dimensionality Reduction

### i) Use PCA to choose the number of dimension we want to keep

Let's look at the number of important dimension by creating an elbow plot and seing when the variance stop decreasing

In [ ]:
sc.tl.pca(adata, svd_solver='arpack') 

variance_ratio = adata.uns['pca']['variance_ratio']

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(variance_ratio) + 1), variance_ratio, 'o-', markersize=4, color='black')
ax.axvline(x=20, color='red', linestyle='--', label='Cutoff : PC 20')
ax.set_yscale('log')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (log scale)')
ax.set_title('Elbow Plot - PCA Variance Ratio')
ax.legend()
plt.tight_layout()
plt.show()

### UMAP

In [ ]:
# Compute neighborhood graph (required before UMAP/tSNE)
# n_pcs match our elbow plot decision : 20
sc.pp.neighbors(adata, n_pcs=20)

# Run UMAP
sc.tl.umap(adata)

# Run t-SNE
sc.tl.tsne(adata, n_pcs=20)

# Run Leiden clustering
sc.tl.leiden(adata, resolution=0.5, flavor='igraph')  # adjust resolution to get more/fewer clusters

# Visualize side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.umap(adata, ax=axes[0], color ='leiden', show=False, title='UMAP')
sc.pl.tsne(adata, ax=axes[1], color ='leiden', show=False, title='tSNE')
plt.tight_layout()
plt.show()

### Let's associate an identity to each of the gene using the marker

Load Cell_marker_Human.xslsx  as a clv as a dataframe and filter the column to only keep cell for the species "Human", 

df_reference_marker_gene = pd.read_excel(""Cell_marker_Human.xlsx)

In [ ]:
df_human = pd.read_excel("Cell_marker_Human.xlsx", engine="openpyxl")


tissues_of_interest = [
    "Ovary",            # primary tumor site
    "Fallopian tube",   # HGSOC origin
    "Peritoneum",       # metastatic site — peritoneal carcinomatosis
    "Ascites",          # peritoneal fluid, very relevant for HGSOC spread
    "Omentum",          # common metastatic deposit site
]

df_tme = df_human[
    df_human["tissue_type"].str.contains(
        "|".join(tissues_of_interest),
        case=False, na=False
    )
]

print(df_tme["tissue_type"].value_counts())

df_ovarian_cancer = df_human[
    df_human["cancer_type"].str.contains("Ovarian|ovarian", na=False)
]
print(df_ovarian_cancer["cancer_type"].value_counts())

# Keep only relevant column :
df_markers = df_ovarian_cancer[["cell_name", "Symbol"]].dropna()
df_markers.columns = ["cell_type", "gene"]

# Keep only markers pressent in the dataset

marker_dict = (
    df_markers.groupby("cell_type")["gene"]
    .apply(list)
    .to_dict()
)

marker_dict_filtered = {
    cell: [g for g in genes if g in adata.var_names]
    for cell, genes in marker_dict.items()
}

tissue_type
Ovary         457
Ascites        20
Peritoneum      6
Omentum         4
Name: count, dtype: int64
cancer_type
Ovarian Cancer                           321
High-grade serous ovarian cancer          34
Ovarian clear cell carcinoma               8
Unspecified Epithelial Ovarian Cancer      7
Serous Ovarian Cancer                      5
Ovarian cancer                             4
Ovarian Adenocarcinoma                     4
Ovarian Teratocarcinoma                    3
Name: count, dtype: int64


NameError: name 'marker_dict' is not defined